In [9]:
# ============================================================
# FULL CLEAN REBUILD — from raw HuggingFace parquet, no dependency
# on the old archive/collection. Run cells in order, top to bottom.
# ============================================================

# %% CELL 1 — Imports + Config
import re
import uuid
import sqlite3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from difflib import SequenceMatcher
from huggingface_hub import hf_hub_download

SAMPLE_SIZE_PER_LANG = 3000
QUERY_TYPE_DIST = {
    "DESCRIPTION": 0.529,
    "NUMERIC": 0.263,
    "ENTITY": 0.089,
    "LOCATION": 0.063,
    "PERSON": 0.056,
}
YEAR_RE = re.compile(r"\b(1[89]\d{2}|20\d{2})\b")

UNIFIED_FILE = "../unified_corpus.parquet"
SQLITE_DB = "native_text_lookup.db"
QDRANT_COLLECTION = "msmarco_english_corpus"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
EMBED_DIM = 384
DEDUP_THRESHOLD = 0.9


print("Config loaded.")


Config loaded.


In [20]:
DATA_DIR= "../MSMARCO-XI/train/hintrain.parquet"

In [10]:
# %% CELL 2 — Download + stream-read helpers
def stream_read_parquet(path: str, batch_size: int = 5000) -> pd.DataFrame:
    pf = pq.ParquetFile(path)
    batches = [batch.to_pandas() for batch in pf.iter_batches(batch_size=batch_size)]
    return pd.concat(batches, ignore_index=True)

def download_language_file(repo_id: str, filename: str) -> str:
    return hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")

print("Download helpers ready.")


Download helpers ready.


In [11]:

# %% CELL 3 — Stratified subsample
def stratified_sample(df: pd.DataFrame, n_total: int, seed: int = 42) -> pd.DataFrame:
    parts = []
    for qtype, frac in QUERY_TYPE_DIST.items():
        n = int(round(n_total * frac))
        pool = df[df["query_type"] == qtype]
        if len(pool) == 0:
            continue
        n = min(n, len(pool))
        parts.append(pool.sample(n=n, random_state=seed))
    return pd.concat(parts, ignore_index=True) if parts else df.sample(
        n=min(n_total, len(df)), random_state=seed
    )

print("Sampling helper ready.")


Sampling helper ready.


In [12]:

# %% CELL 4 — Explode passages (keeps native text)
def explode_msmarco_xi(df: pd.DataFrame, lang_code: str) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        p = r["passages"]
        eng_passages = p["English_passages"]
        trans_passages = p["Translated_passages"]
        is_sel = p["is_selected"]
        for i, (en_p, tr_p, sel) in enumerate(zip(eng_passages, trans_passages, is_sel)):
            rows.append({
                "text_en": en_p,
                "passage_id": f"{lang_code}_{r['query_id']}_{i}",
                "query_id": int(r["query_id"]),
                "lang": lang_code,
                "query_type": r["query_type"],
                "source_dataset": "MSMARCO-XI",
                "is_selected": bool(sel),
                "text": tr_p,
                "query": r["query"],
                "answer": r["Answer"],
                "query_en": r["Eng_Query"],
                "answer_en": r["Eng_Answer"],
            })
    return pd.DataFrame(rows)

print("Explode function ready.")


Explode function ready.


In [13]:

# %% CELL 5 — Preprocessing (cleanup BEFORE chunking)
import html

def preprocess_text(text: str) -> str:
    if not text:
        return ""
    text = html.unescape(text)
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"www\.\S+", "", text)
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\s*\.\s*'\s*", ". ", text)
    text = re.sub(r"…", " ", text)
    text = re.sub(r"(\d)\s*…\s*(\d)", r"\1\2", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_valid_row(text: str, min_chars: int = 15) -> bool:
    return bool(text) and len(text.strip()) >= min_chars

def preprocess_corpus(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text_en"] = df["text_en"].apply(preprocess_text)
    df["text"] = df["text"].apply(preprocess_text)
    before = len(df)
    df = df[df["text_en"].apply(is_valid_row)].reset_index(drop=True)
    print(f"Validity filter: {before:,} -> {len(df):,} rows")
    return df

print("Preprocessing functions ready.")


Preprocessing functions ready.


In [14]:

# %% CELL 6 — Enrich: regex years + spaCy NER
def enrich(df: pd.DataFrame) -> pd.DataFrame:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    year_mentions, ppl, orgs, locs = [], [], [], []
    for doc_text in nlp.pipe(df["text_en"].fillna("").tolist(), batch_size=256):
        year_mentions.append([int(y) for y in YEAR_RE.findall(doc_text.text)])
        ppl.append([e.text for e in doc_text.ents if e.label_ == "PERSON"])
        orgs.append([e.text for e in doc_text.ents if e.label_ == "ORG"])
        locs.append([e.text for e in doc_text.ents if e.label_ in ("GPE", "LOC")])
    df["year_mentions"] = year_mentions
    df["entities_people"] = ppl
    df["entities_orgs"] = orgs
    df["entities_locations"] = locs
    return df

print("Enrich function ready.")

Enrich function ready.


In [15]:

# %% CELL 7 — Chunking: 3 strategies (semantic = fixed version, 128/256/320)
def chunk_passage_native(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["chunk_strategy"] = "passage_native"
    out["chunk_id"] = out["passage_id"] + "_native"
    return out

def chunk_fixed_overlap(df: pd.DataFrame, chunk_tokens: int = 256, overlap: float = 0.2) -> pd.DataFrame:
    rows = []
    step = int(chunk_tokens * (1 - overlap))
    for _, r in df.iterrows():
        words = (r["text_en"] or "").split()
        if len(words) <= chunk_tokens:
            spans = [words]
        else:
            spans = [words[i:i + chunk_tokens] for i in range(0, len(words), step) if words[i:i + chunk_tokens]]
        for j, span in enumerate(spans):
            new_row = r.to_dict()
            new_row["text_en"] = " ".join(span)
            new_row["chunk_strategy"] = "fixed_overlap"
            new_row["chunk_id"] = f"{r['passage_id']}_fixed_{j}"
            rows.append(new_row)
    return pd.DataFrame(rows)

def chunk_semantic(df: pd.DataFrame, model, similarity_threshold: float = 0.75,
                    target_tokens: int = 256, min_tokens: int = 128, max_tokens: int = 320) -> pd.DataFrame:
    import re as _re

    def word_count(sents):
        return sum(len(s.split()) for s in sents)

    rows = []
    for _, r in df.iterrows():
        sents = [s.strip() for s in _re.split(r"(?<=[.!?])\s+", r["text_en"] or "") if s.strip()]
        if not sents:
            continue
        if len(sents) == 1:
            groups = [sents]
        else:
            embs = model.encode(sents, normalize_embeddings=True)
            groups, current = [], [sents[0]]
            for i in range(1, len(sents)):
                sim = float(np.dot(embs[i - 1], embs[i]))
                current_len = word_count(current)
                force_split = current_len >= max_tokens
                natural_split = (sim < similarity_threshold) and (current_len >= target_tokens * 0.7)
                if force_split or natural_split:
                    groups.append(current)
                    current = []
                current.append(sents[i])
            groups.append(current)

            merged = []
            i = 0
            while i < len(groups):
                g = groups[i]
                if word_count(g) < min_tokens:
                    if i + 1 < len(groups):
                        groups[i + 1] = g + groups[i + 1]
                    elif merged:
                        merged[-1] = merged[-1] + g
                    else:
                        merged.append(g)
                    i += 1
                    continue
                merged.append(g)
                i += 1
            groups = merged

        for j, g in enumerate(groups):
            if not g:
                continue
            new_row = r.to_dict()
            new_row["text_en"] = " ".join(g)
            new_row["chunk_strategy"] = "semantic"
            new_row["chunk_id"] = f"{r['passage_id']}_sem_{j}"
            new_row["chunk_word_count"] = word_count(g)
            rows.append(new_row)
    return pd.DataFrame(rows)

def tag_metadata_aware(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["has_year"] = df["year_mentions"].apply(lambda y: len(y) > 0)
    df["has_person"] = df["entities_people"].apply(lambda p: len(p) > 0)
    df["has_org"] = df["entities_orgs"].apply(lambda o: len(o) > 0)
    df["has_location"] = df["entities_locations"].apply(lambda l: len(l) > 0)
    return df

print("Chunking functions ready.")

Chunking functions ready.


In [16]:

# %% CELL 8 — Dedup with passage_native-first priority
STRATEGY_PRIORITY = {"passage_native": 0, "semantic": 1, "fixed_overlap": 2}

def dedupe_chunks(df: pd.DataFrame, threshold: float = DEDUP_THRESHOLD) -> pd.DataFrame:
    keep_rows = []
    before = len(df)
    for passage_id, group in df.groupby("passage_id"):
        group_sorted = group.copy()
        group_sorted["_priority"] = group_sorted["chunk_strategy"].map(STRATEGY_PRIORITY).fillna(99)
        group_sorted = group_sorted.sort_values("_priority")
        seen_texts = []
        for _, row in group_sorted.iterrows():
            text = row["text_en"] or ""
            if any(SequenceMatcher(None, text, s).ratio() > threshold for s in seen_texts):
                continue
            seen_texts.append(text)
            keep_rows.append(row.drop("_priority"))
    result = pd.DataFrame(keep_rows)
    after = len(result)
    print(f"Dedup: {before:,} -> {after:,} chunks ({before - after:,} removed, "
          f"{(before - after) / before * 100:.1f}% reduction)")
    print(result["chunk_strategy"].value_counts())
    return result

print("Dedup function ready.")

Dedup function ready.


In [17]:

# %% CELL 9 — SQLite native lookup + archive writer
def build_native_lookup(df: pd.DataFrame, db_path: str = SQLITE_DB):
    native_lookup = df.drop_duplicates(subset=["passage_id"])[
        ["passage_id", "query_id", "lang", "text", "query", "answer"]
    ]
    conn = sqlite3.connect(db_path)
    native_lookup.to_sql("passages", conn, if_exists="replace", index=False)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_passage_id ON passages(passage_id)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_query_id ON passages(query_id)")
    conn.commit()
    conn.close()
    print(f"SQLite native lookup: {len(native_lookup):,} unique passages -> {db_path}")

def write_unified_file(df: pd.DataFrame, path: str = UNIFIED_FILE):
    df.to_parquet(path, index=False)
    print(f"Archive written: {len(df):,} rows -> {path}")

print("SQLite/archive writers ready.")

SQLite/archive writers ready.


In [18]:

# %% CELL 10 — Embed + upsert to Qdrant (fresh collection, indexes first)
QDRANT_PAYLOAD_FIELDS = [
    "text_en", "passage_id", "query_id", "lang", "query_type", "source_dataset",
    "chunk_strategy", "chunk_id", "has_year", "has_person", "has_org", "has_location",
    "year_mentions", "chunk_word_count",
]

def embed_and_upsert(df: pd.DataFrame, qdrant_url: str = "http://localhost:6333"):
    from sentence_transformers import SentenceTransformer
    import os
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType

    df = df[df["text_en"].notna() & (df["text_en"].str.strip() != "")].reset_index(drop=True)

    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

    model = SentenceTransformer(
        EMBED_MODEL,
        local_files_only=True
    )
    client = QdrantClient(url=qdrant_url)

    if client.collection_exists(QDRANT_COLLECTION):
        client.delete_collection(collection_name=QDRANT_COLLECTION)

    client.create_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )

    index_fields = [
        ("lang", PayloadSchemaType.KEYWORD),
        ("query_type", PayloadSchemaType.KEYWORD),
        ("chunk_strategy", PayloadSchemaType.KEYWORD),
        ("source_dataset", PayloadSchemaType.KEYWORD),
        ("has_year", PayloadSchemaType.BOOL),
        ("has_person", PayloadSchemaType.BOOL),
        ("has_org", PayloadSchemaType.BOOL),
        ("has_location", PayloadSchemaType.BOOL),
        ("year_mentions", PayloadSchemaType.INTEGER),
    ]
    for field, schema in index_fields:
        client.create_payload_index(collection_name=QDRANT_COLLECTION, field_name=field, field_schema=schema)
    print(f"Created {len(index_fields)} payload indexes.")

    texts = df["text_en"].tolist()
    vectors = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

    points = []
    for vec, (_, row) in zip(vectors, df.iterrows()):
        payload = {}
        for k in QDRANT_PAYLOAD_FIELDS:
            if k not in row:
                continue
            v = row[k]
            if isinstance(v, list) or pd.notna(v):
                payload[k] = v
        points.append(PointStruct(id=str(uuid.uuid4()), vector=vec.tolist(), payload=payload))

    B = 256
    for i in range(0, len(points), B):
        client.upsert(collection_name=QDRANT_COLLECTION, points=points[i:i + B])

    print(f"Upserted {len(points):,} points into '{QDRANT_COLLECTION}'")
    return client, model

print("Embed/upsert function ready.")

Embed/upsert function ready.


In [21]:
# %% CELL 11 — RUN: download + sample + normalize Hindi
hi_path = DATA_DIR
hi_df = stream_read_parquet(hi_path)
hi_df = stratified_sample(hi_df, SAMPLE_SIZE_PER_LANG)
hi_norm = explode_msmarco_xi(hi_df, "hi")
print(f"Hindi normalized: {len(hi_norm):,} passage rows")

Hindi normalized: 29,905 passage rows


In [22]:
# %% CELL 12 — Combine (add Odia/Telugu here later if needed) + preprocess
unified = pd.concat([hi_norm], ignore_index=True)
unified = preprocess_corpus(unified)
print(f"Unified after preprocessing: {len(unified):,} rows")

Validity filter: 29,905 -> 29,904 rows
Unified after preprocessing: 29,904 rows


In [23]:
# %% CELL 13 — Enrich
unified = enrich(unified)
print("Enrichment done.")

Enrichment done.


In [25]:
# %% CELL 14 — Chunk (all 3 strategies)
native = chunk_passage_native(unified)
fixed = chunk_fixed_overlap(unified)

from sentence_transformers import SentenceTransformer
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

model = SentenceTransformer(
    EMBED_MODEL,
    local_files_only=True
)
sem_model = model
semantic = chunk_semantic(unified, sem_model)

print(f"native: {len(native):,} | fixed: {len(fixed):,} | semantic: {len(semantic):,} (all pre-dedup)")

all_chunks = pd.concat([native, fixed, semantic], ignore_index=True)

native: 29,904 | fixed: 29,904 | semantic: 29,904 (all pre-dedup)


In [26]:
# %% CELL 15 — Tag metadata + dedup
all_chunks = tag_metadata_aware(all_chunks)
all_chunks = dedupe_chunks(all_chunks)

Dedup: 89,712 -> 29,904 chunks (59,808 removed, 66.7% reduction)
chunk_strategy
passage_native    29904
Name: count, dtype: int64


In [27]:
# %% CELL 16 — Write archive + SQLite lookup
write_unified_file(all_chunks)
build_native_lookup(all_chunks)

Archive written: 29,904 rows -> unified_corpus.parquet
SQLite native lookup: 29,904 unique passages -> native_text_lookup.db


In [28]:
# %% CELL 17 — Embed + upsert to Qdrant
qdrant_client, embed_model = embed_and_upsert(all_chunks)


Created 9 payload indexes.


Batches:   0%|          | 0/468 [00:00<?, ?it/s]

Upserted 29,904 points into 'msmarco_english_corpus'


In [29]:
# %% CELL 18 — Verify
info = qdrant_client.get_collection(QDRANT_COLLECTION)
print(info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=29904 segments_count=6 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, pre

In [31]:

# %% CELL 19 — Sanity check
def retrieve_top_k(query_text, top_k=5, model=embed_model, qdrant_client=qdrant_client):
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    qvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    # Newer qdrant-client (>=1.10ish) removed .search() in favor of .query_points()
    try:
        response = qdrant_client.query_points(collection_name=QDRANT_COLLECTION, query=qvec, limit=top_k)
        results = response.points
    except AttributeError:
        # fallback for older qdrant-client versions that still have .search()
        results = qdrant_client.search(collection_name=QDRANT_COLLECTION, query_vector=qvec, limit=top_k)

    for i, r in enumerate(results, 1):
        print(f"{i}. Score: {r.score:.4f} | Strategy: {r.payload.get('chunk_strategy')} | "
              f"Words: {r.payload.get('chunk_word_count', '?')}")
        print(f"   {r.payload.get('text_en')[:200]}")
    return results

retrieve_top_k("what is Error Free Synonym", top_k=5)

1. Score: 0.7324 | Strategy: passage_native | Words: ?
   Synonym Discussion of ERROR. error, mistake, and blunder mean an act or statement that is not right or true or proper.
2. Score: 0.6464 | Strategy: passage_native | Words: ?
   1 First, present your prospective employer with a concise, error-free resume. 2 The editors are to be congratulated for an error-free, genuinely erudite text. 3 Granted, there are no error-free method
3. Score: 0.6339 | Strategy: passage_native | Words: ?
   1 part of a statement that is not correct 1. 2 the book was full of errors 1. ( 3 computer science) the occurrence of an incorrect result produced by a computer 1. 4 A value or condition that is not c
4. Score: 0.6313 | Strategy: passage_native | Words: ?
   uk ​ /ˈmʌm.bəl/ us ​ /ˈmʌm.bəl/. B2 to speak quietly and in a way that is not clear so that the words are difficult to understand: She mumbled something about being too busy. [ + speech ] I'm sorry, h
5. Score: 0.6243 | Strategy: passage_native | 

[ScoredPoint(id='ee3fd442-dba1-4966-9501-32d1322a2e44', version=54, score=0.7324295, payload={'text_en': 'Synonym Discussion of ERROR. error, mistake, and blunder mean an act or statement that is not right or true or proper.', 'passage_id': 'hi_2923_6', 'query_id': 2923, 'lang': 'hi', 'query_type': 'DESCRIPTION', 'source_dataset': 'MSMARCO-XI', 'chunk_strategy': 'passage_native', 'chunk_id': 'hi_2923_6_native', 'has_year': False, 'has_person': False, 'has_org': False, 'has_location': False, 'year_mentions': []}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='69932f62-7ccd-46a1-be87-c190c970af03', version=54, score=0.6463635, payload={'text_en': '1 First, present your prospective employer with a concise, error-free resume. 2 The editors are to be congratulated for an error-free, genuinely erudite text. 3 Granted, there are no error-free methods of tabulating votes.', 'passage_id': 'hi_2923_1', 'query_id': 2923, 'lang': 'hi', 'query_type': 'DESCRIPTION', 'source_dataset

In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

from fastembed import SparseTextEmbedding
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25", cache_dir="../fastembed_cache")
print("sparse_model loaded.")

In [49]:
from huggingface_hub import snapshot_download

path = snapshot_download(repo_id="Qdrant/bm25", cache_dir="../hf_cache")
print("Downloaded to:", path)

Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

italian.txt: 0.00B [00:00, ?B/s]

hebrew.txt: 0.00B [00:00, ?B/s]

hungarian.txt: 0.00B [00:00, ?B/s]

hinglish.txt: 0.00B [00:00, ?B/s]

...

stopwords.txt:   0%|          | 0.00/936 [00:00<?, ?B/s]

slovene.txt: 0.00B [00:00, ?B/s]

turkish.txt:   0%|          | 0.00/260 [00:00<?, ?B/s]

tajik.txt: 0.00B [00:00, ?B/s]

Downloaded to: ./hf_cache\models--Qdrant--bm25\snapshots\e499a1f8d6bec960aab5533a0941bf914e70faf9


In [50]:
from fastembed import SparseTextEmbedding

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25",
    specific_model_path="../hf_cache/models--Qdrant--bm25/snapshots/e499a1f8d6bec960aab5533a0941bf914e70faf9"
)
print("sparse_model loaded.")

sparse_model loaded.


In [52]:
info = qdrant_client.get_collection(QDRANT_COLLECTION)
print("Dense vectors config:", info.config.params.vectors)
print("Sparse vectors config:", info.config.params.sparse_vectors)

Dense vectors config: size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None
Sparse vectors config: None


In [53]:
def embed_and_upsert_hybrid(df: pd.DataFrame, qdrant_url: str = "http://localhost:6333"):
    from sentence_transformers import SentenceTransformer
    from qdrant_client import QdrantClient
    from qdrant_client.models import (
        Distance, VectorParams, SparseVectorParams, SparseVector,
        PointStruct, PayloadSchemaType
    )

    df = df[df["text_en"].notna() & (df["text_en"].str.strip() != "")].reset_index(drop=True)

    dense_model = SentenceTransformer(EMBED_MODEL)
    client = QdrantClient(url=qdrant_url)

    if client.collection_exists(QDRANT_COLLECTION):
        client.delete_collection(collection_name=QDRANT_COLLECTION)

    client.create_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config={"dense": VectorParams(size=EMBED_DIM, distance=Distance.COSINE)},
        sparse_vectors_config={"sparse": SparseVectorParams()},
    )

    index_fields = [
        ("lang", PayloadSchemaType.KEYWORD),
        ("query_type", PayloadSchemaType.KEYWORD),
        ("chunk_strategy", PayloadSchemaType.KEYWORD),
        ("source_dataset", PayloadSchemaType.KEYWORD),
        ("has_year", PayloadSchemaType.BOOL),
        ("has_person", PayloadSchemaType.BOOL),
        ("has_org", PayloadSchemaType.BOOL),
        ("has_location", PayloadSchemaType.BOOL),
        ("year_mentions", PayloadSchemaType.INTEGER),
    ]
    for field, schema in index_fields:
        client.create_payload_index(collection_name=QDRANT_COLLECTION, field_name=field, field_schema=schema)
    print(f"Created {len(index_fields)} payload indexes.")

    texts = df["text_en"].tolist()
    print(f"Encoding {len(texts):,} texts (dense)...")
    dense_vectors = dense_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

    print(f"Encoding {len(texts):,} texts (sparse)...")
    sparse_vectors = list(sparse_model.embed(texts, batch_size=64))

    points = []
    for dvec, svec, (_, row) in zip(dense_vectors, sparse_vectors, df.iterrows()):
        payload = {}
        for k in QDRANT_PAYLOAD_FIELDS:
            if k not in row:
                continue
            v = row[k]
            if isinstance(v, list) or pd.notna(v):
                payload[k] = v
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector={
                    "dense": dvec.tolist(),
                    "sparse": SparseVector(
                        indices=svec.indices.tolist(),
                        values=svec.values.tolist()
                    ),
                },
                payload=payload,
            )
        )

    B = 256
    total_batches = (len(points) + B - 1) // B
    for i in range(0, len(points), B):
        client.upsert(collection_name=QDRANT_COLLECTION, points=points[i:i + B])
        print(f"  Upserted batch {i // B + 1}/{total_batches}")

    print(f"Upserted {len(points):,} hybrid points into '{QDRANT_COLLECTION}'")
    return client, dense_model, sparse_model

In [54]:
qdrant_client, embed_model, sparse_model = embed_and_upsert_hybrid(all_chunks)

Created 9 payload indexes.
Encoding 29,904 texts (dense)...


Batches:   0%|          | 0/468 [00:00<?, ?it/s]

Encoding 29,904 texts (sparse)...
  Upserted batch 1/117
  Upserted batch 2/117
  Upserted batch 3/117
  Upserted batch 4/117
  Upserted batch 5/117
  Upserted batch 6/117
  Upserted batch 7/117
  Upserted batch 8/117
  Upserted batch 9/117
  Upserted batch 10/117
  Upserted batch 11/117
  Upserted batch 12/117
  Upserted batch 13/117
  Upserted batch 14/117
  Upserted batch 15/117
  Upserted batch 16/117
  Upserted batch 17/117
  Upserted batch 18/117
  Upserted batch 19/117
  Upserted batch 20/117
  Upserted batch 21/117
  Upserted batch 22/117
  Upserted batch 23/117
  Upserted batch 24/117
  Upserted batch 25/117
  Upserted batch 26/117
  Upserted batch 27/117
  Upserted batch 28/117
  Upserted batch 29/117
  Upserted batch 30/117
  Upserted batch 31/117
  Upserted batch 32/117
  Upserted batch 33/117
  Upserted batch 34/117
  Upserted batch 35/117
  Upserted batch 36/117
  Upserted batch 37/117
  Upserted batch 38/117
  Upserted batch 39/117
  Upserted batch 40/117
  Upserted batc

In [55]:
info = qdrant_client.get_collection(QDRANT_COLLECTION)
print("Sparse vectors config:", info.config.params.sparse_vectors)

Sparse vectors config: {'sparse': SparseVectorParams(index=None, modifier=None)}


In [56]:
# %% CELL 19-SPARSE-ONLY — separate sparse-only retrieval, doesn't touch existing code

def retrieve_top_k_sparse_only(query_text, top_k=5, sparse_model=sparse_model, qdrant_client=qdrant_client):
    import time
    from qdrant_client.models import SparseVector

    t0 = time.perf_counter()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
        using="sparse",
        limit=top_k,
    )
    t2 = time.perf_counter()

    return {
        "results": response.points,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t2 - t0) * 1000,
    }

# sanity check
r = retrieve_top_k_sparse_only("what is Error Free Synonym", top_k=5)
for i, res in enumerate(r["results"], 1):
    print(f"{i}. Score: {res.score:.4f} | {res.payload.get('text_en')[:150]}")

1. Score: 6.5473 | 1 First, present your prospective employer with a concise, error-free resume. 2 The editors are to be congratulated for an error-free, genuinely erudi
2. Score: 6.0009 | Shush.se - Watch The Office Season 8 Episode: 5 – Spooked | Free Online Streaming |. Shush.se - Watch The Office Season 8 Episode: 5 – Spooked | Free 
3. Score: 5.9051 | Synonym Discussion of ERROR. error, mistake, and blunder mean an act or statement that is not right or true or proper.
4. Score: 5.5031 | Target cell definition at Dictionary.com, a free online dictionary with pronunciation, synonyms and translation. Look it up now!
5. Score: 5.2818 | Synonym of Awkward silence: English Wikipedia - The Free Encyclopedia Awkward silence An awkward silence is an uncomfortable pause in a conversation o


In [57]:
# %% CELL 19-HYBRID-ONLY — separate hybrid (dense+sparse fused) retrieval, doesn't touch existing code

def retrieve_top_k_hybrid_only(query_text, top_k=5, model=embed_model, sparse_model=sparse_model,
                                qdrant_client=qdrant_client):
    import time
    from qdrant_client.models import SparseVector, Prefetch, FusionQuery, Fusion

    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    dvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        prefetch=[
            Prefetch(query=dvec, using="dense", limit=20),
            Prefetch(
                query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
                using="sparse",
                limit=20,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
    )
    t2 = time.perf_counter()

    return {
        "results": response.points,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t2 - t0) * 1000,
    }

# sanity check
r = retrieve_top_k_hybrid_only("what is Error Free Synonym", top_k=5)
for i, res in enumerate(r["results"], 1):
    print(f"{i}. Score: {res.score:.4f} | {res.payload.get('text_en')[:150]}")

1. Score: 0.8333 | 1 First, present your prospective employer with a concise, error-free resume. 2 The editors are to be congratulated for an error-free, genuinely erudi
2. Score: 0.7500 | Synonym Discussion of ERROR. error, mistake, and blunder mean an act or statement that is not right or true or proper.
3. Score: 0.3333 | Shush.se - Watch The Office Season 8 Episode: 5 – Spooked | Free Online Streaming |. Shush.se - Watch The Office Season 8 Episode: 5 – Spooked | Free 
4. Score: 0.2500 | 1 part of a statement that is not correct 1. 2 the book was full of errors 1. ( 3 computer science) the occurrence of an incorrect result produced by 
5. Score: 0.2000 | uk ​ /ˈmʌm.bəl/ us ​ /ˈmʌm.bəl/. B2 to speak quietly and in a way that is not clear so that the words are difficult to understand: She mumbled somethi


In [32]:
# %% CELL 22 — retrieve_top_k_silent: timed retrieval helper (no print, returns timing dict)
# Required by CELL 23b. Mirrors retrieve_top_k but returns structured timing + results
# instead of printing, so it can be called 60x in a loop without flooding output.

import time

def retrieve_top_k_silent(query_text, top_k=5, model=None, qdrant_client=None,
                           collection_name=None):
    """Returns {results, embedding_ms, qdrant_ms, dedup_ms, total_ms}.
    dedup_ms is 0 here since dedup already happened at ingestion time --
    kept as a field for schema consistency with earlier timing logs."""
    model = model or embed_model
    qdrant_client = qdrant_client or globals().get("qdrant_client")
    collection_name = collection_name or QDRANT_COLLECTION

    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    qvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    t1 = time.perf_counter()

    try:
        response = qdrant_client.query_points(collection_name=collection_name, query=qvec, limit=top_k)
        results = response.points
    except AttributeError:
        results = qdrant_client.search(collection_name=collection_name, query_vector=qvec, limit=top_k)
    t2 = time.perf_counter()

    # dedup already done at ingestion time; this stage is a no-op at query time,
    # kept for schema parity with the earlier live-dedup benchmark format
    t3 = time.perf_counter()

    return {
        "results": results,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": (t3 - t2) * 1000,
        "total_ms": (t3 - t0) * 1000,
    }

print("retrieve_top_k_silent ready.")

# ============================================================
# CELL 23 and CELL 23b go here exactly as you already have them
# (test-set construction from disk, then the 60-query benchmark loop).
# No changes needed to those two cells -- they're correct as written.
# ============================================================

retrieve_top_k_silent ready.


In [33]:
# %% CELL 23 — 60-Question Benchmark, sourced fresh from disk (parquet + SQLite)

import sqlite3
import numpy as np
import pandas as pd
import random

N_TEST_QUERIES = 60
RANDOM_SEED = 42
TOP_K = 5

random.seed(RANDOM_SEED)

# ---------------------------------------------------------
# 1. Load source of truth fresh from disk
# ---------------------------------------------------------
archive_df = pd.read_parquet(UNIFIED_FILE)
print(f"Loaded archive from disk: {len(archive_df):,} rows -> {UNIFIED_FILE}")

conn = sqlite3.connect(SQLITE_DB)
sqlite_df = pd.read_sql_query("SELECT * FROM passages", conn)
conn.close()
print(f"Loaded SQLite lookup: {len(sqlite_df):,} rows -> {SQLITE_DB}")

# ---------------------------------------------------------
# 2. Cross-check sync between archive and SQLite
# ---------------------------------------------------------
archive_qids = set(archive_df["query_id"].unique())
sqlite_qids = set(sqlite_df["query_id"].unique())

missing_in_sqlite = archive_qids - sqlite_qids
missing_in_archive = sqlite_qids - archive_qids

print(f"query_ids in archive: {len(archive_qids):,}")
print(f"query_ids in SQLite : {len(sqlite_qids):,}")
print(f"In archive but missing from SQLite: {len(missing_in_sqlite)}")
print(f"In SQLite but missing from archive: {len(missing_in_archive)}")

# ---------------------------------------------------------
# 3. Build 60-query test set from validated query_ids
# ---------------------------------------------------------
valid_qids = archive_qids & sqlite_qids

qa_pool = (
    archive_df[
        archive_df["query_en"].notna()
        & (archive_df["query_en"].str.strip() != "")
        & (archive_df["query_id"].isin(valid_qids))
    ]
    .drop_duplicates(subset=["query_id"])
    [["query_id", "query_en", "query_type"]]
)

if len(qa_pool) < N_TEST_QUERIES:
    raise ValueError(f"Only {len(qa_pool)} valid unique English queries available, need {N_TEST_QUERIES}.")

test_set = qa_pool.sample(n=N_TEST_QUERIES, random_state=RANDOM_SEED).reset_index(drop=True)

# ---------------------------------------------------------
# 4. Ground-truth passage_ids per query_id
# ---------------------------------------------------------
gt_map = (
    archive_df[archive_df["is_selected"] == True]
    .groupby("query_id")["passage_id"]
    .apply(set)
    .to_dict()
)

sqlite_passage_ids = set(sqlite_df["passage_id"].unique())
for _, row in test_set.iterrows():
    qid = row["query_id"]
    gt_ids = gt_map.get(qid, set())
    missing = gt_ids - sqlite_passage_ids
    if missing:
        print(f"WARNING: query_id={qid} has ground-truth passage_id(s) {missing} not in SQLite (dropped during dedup).")

print(f"\nTest set: {len(test_set)} queries")
print(test_set[["query_id", "query_type", "query_en"]].to_string(index=False))

Loaded archive from disk: 29,904 rows -> unified_corpus.parquet
Loaded SQLite lookup: 29,904 rows -> native_text_lookup.db
query_ids in archive: 3,000
query_ids in SQLite : 3,000
In archive but missing from SQLite: 0
In SQLite but missing from archive: 0

Test set: 60 queries
 query_id  query_type                                                                                  query_en
   592568 DESCRIPTION                                         what causes the convection currents in the mantle
   388153 DESCRIPTION                                                how were the first people to use the flute
   597804      ENTITY                                                                      what color is cognac
   109359     NUMERIC                                                                     cost to fedex package
   839797     NUMERIC                                                      what is the population of big sky mt
   362791 DESCRIPTION                              

In [34]:
# %% CELL 23b — Run retrieval + timing + quality on the 60-query test set

records = []
for i, row in test_set.iterrows():
    qid, qtext, qtype = row["query_id"], row["query_en"], row["query_type"]

    r = retrieve_top_k_silent(qtext, top_k=TOP_K)
    retrieved_ids = [res.payload.get("passage_id") for res in r["results"]]
    scores = [res.score for res in r["results"]]

    relevant_ids = gt_map.get(qid, set())
    hit = any(pid in relevant_ids for pid in retrieved_ids)

    rr = 0.0
    for rank, pid in enumerate(retrieved_ids, 1):
        if pid in relevant_ids:
            rr = 1.0 / rank
            break

    native_rows = sqlite_df[sqlite_df["query_id"] == qid]
    has_native = len(native_rows) > 0

    records.append({
        "query_idx": i + 1,
        "query_id": qid,
        "query_type": qtype,
        "query_en": qtext,
        "embedding_ms": r["embedding_ms"],
        "qdrant_ms": r["qdrant_ms"],
        "dedup_ms": r["dedup_ms"],
        "total_ms": r["total_ms"],
        "top1_score": scores[0] if scores else None,
        "n_results": len(retrieved_ids),
        "retrieved_passage_ids": retrieved_ids,
        "relevant_passage_ids": list(relevant_ids),
        "hit@k": hit,
        "reciprocal_rank": rr,
        "has_native_row_in_sqlite": has_native,
    })

    if scores:
        print(f"[{i+1:2d}/{N_TEST_QUERIES}] total={r['total_ms']:7.2f}ms  "
              f"hit={hit}  top1_score={scores[0]:.4f}  native_ok={has_native}")
    else:
        print(f"[{i+1:2d}/{N_TEST_QUERIES}] no results")

results_df = pd.DataFrame(records)

# ---------------------------------------------------------
# Percentiles
# ---------------------------------------------------------
def compute_percentiles(series, percentiles=(30, 50, 70, 90, 99)):
    return {f"p{p}": round(np.percentile(series, p), 2) for p in percentiles}

stages = ["embedding_ms", "qdrant_ms", "dedup_ms", "total_ms"]
summary_rows = []
for stage in stages:
    pct = compute_percentiles(results_df[stage])
    summary_rows.append({
        "stage": stage, **pct,
        "mean": round(results_df[stage].mean(), 2),
        "min": round(results_df[stage].min(), 2),
        "max": round(results_df[stage].max(), 2),
        "std": round(results_df[stage].std(), 2),
    })
timing_summary_df = pd.DataFrame(summary_rows)

valid_gt = results_df[results_df["relevant_passage_ids"].apply(len) > 0]
quality_summary = pd.DataFrame([{
    "n_queries": len(results_df),
    "n_with_valid_gt": len(valid_gt),
    "top_k": TOP_K,
    f"hit_rate@{TOP_K}_all": round(results_df["hit@k"].mean(), 4),
    f"hit_rate@{TOP_K}_valid_gt_only": round(valid_gt["hit@k"].mean(), 4) if len(valid_gt) else None,
    "MRR_all": round(results_df["reciprocal_rank"].mean(), 4),
    "MRR_valid_gt_only": round(valid_gt["reciprocal_rank"].mean(), 4) if len(valid_gt) else None,
    "avg_top1_score": round(results_df["top1_score"].mean(), 4),
}])

by_type = results_df.groupby("query_type").agg(
    n=("query_idx", "count"),
    hit_rate=("hit@k", "mean"),
    mrr=("reciprocal_rank", "mean"),
    avg_total_ms=("total_ms", "mean"),
).round(4).reset_index()

print("\n========== TIMING PERCENTILES (ms) — 60 queries ==========")
print(timing_summary_df.to_string(index=False))
print("\n========== RETRIEVAL QUALITY (disk-validated) ==========")
print(quality_summary.to_string(index=False))
print("\n========== BREAKDOWN BY QUERY TYPE ==========")
print(by_type.to_string(index=False))

results_df.drop(columns=["retrieved_passage_ids", "relevant_passage_ids"]).to_csv(
    "benchmark_60q_details.csv", index=False
)
timing_summary_df.to_csv("benchmark_60q_timing_percentiles.csv", index=False)
quality_summary.to_csv("benchmark_60q_quality.csv", index=False)
by_type.to_csv("benchmark_60q_by_query_type.csv", index=False)

print("\nSaved:")
print(" - benchmark_60q_details.csv")
print(" - benchmark_60q_timing_percentiles.csv")
print(" - benchmark_60q_quality.csv")
print(" - benchmark_60q_by_query_type.csv")

[ 1/60] total= 107.07ms  hit=True  top1_score=0.9108  native_ok=True
[ 2/60] total=  64.82ms  hit=False  top1_score=0.7768  native_ok=True
[ 3/60] total=  61.77ms  hit=True  top1_score=0.8469  native_ok=True
[ 4/60] total=  60.73ms  hit=True  top1_score=0.8455  native_ok=True
[ 5/60] total=  79.90ms  hit=True  top1_score=0.8556  native_ok=True
[ 6/60] total=  56.83ms  hit=False  top1_score=0.8295  native_ok=True
[ 7/60] total=  45.98ms  hit=True  top1_score=0.8906  native_ok=True
[ 8/60] total=  65.04ms  hit=True  top1_score=0.8752  native_ok=True
[ 9/60] total=  61.03ms  hit=False  top1_score=0.7057  native_ok=True
[10/60] total=  76.70ms  hit=False  top1_score=0.7815  native_ok=True
[11/60] total=  76.32ms  hit=True  top1_score=0.8480  native_ok=True
[12/60] total=  74.53ms  hit=True  top1_score=0.8058  native_ok=True
[13/60] total=  81.72ms  hit=True  top1_score=0.8258  native_ok=True
[14/60] total=  67.64ms  hit=True  top1_score=0.7850  native_ok=True
[15/60] total= 119.47ms  hit=F

In [58]:
# %% CELL 24 — Identify queries with poor retrieval results

# 1. Complete misses (correct passage not in top-k at all)
misses_df = results_df[
    (results_df["hit@k"] == False) &
    (results_df["relevant_passage_ids"].apply(len) > 0)  # only count queries with valid ground truth
].copy()

# 2. Low top1_score hits (found it, but confidence was weak)
LOW_SCORE_THRESHOLD = 0.75
low_score_df = results_df[
    (results_df["top1_score"] < LOW_SCORE_THRESHOLD) &
    (results_df["top1_score"].notna())
].copy()

# 3. Hits but ranked low (found it, but not near top — bad MRR)
low_rank_df = results_df[
    (results_df["hit@k"] == True) &
    (results_df["reciprocal_rank"] < 0.5)  # i.e. correct answer wasn't rank 1 or 2
].copy()

print(f"Total queries: {len(results_df)}")
print(f"Complete misses (valid GT, not found): {len(misses_df)}")
print(f"Low top1_score (< {LOW_SCORE_THRESHOLD}): {len(low_score_df)}")
print(f"Hit but low rank (found, RR < 0.5): {len(low_rank_df)}")

print("\n========== COMPLETE MISSES ==========")
if len(misses_df):
    print(misses_df[[
        "query_idx", "query_id", "query_type", "query_en",
        "top1_score", "retrieved_passage_ids", "relevant_passage_ids"
    ]].to_string(index=False))
else:
    print("None ")

print("\n========== LOW top1_score (weak semantic match) ==========")
if len(low_score_df):
    print(low_score_df[[
        "query_idx", "query_id", "query_type", "query_en", "top1_score"
    ]].sort_values("top1_score").to_string(index=False))
else:
    print("None ")

print("\n========== HIT BUT POORLY RANKED ==========")
if len(low_rank_df):
    print(low_rank_df[[
        "query_idx", "query_id", "query_type", "query_en",
        "reciprocal_rank", "retrieved_passage_ids", "relevant_passage_ids"
    ]].sort_values("reciprocal_rank").to_string(index=False))
else:
    print("None ")

# Save for inspection
bad_queries_df = pd.concat([
    misses_df.assign(issue="complete_miss"),
    low_score_df.assign(issue="low_top1_score"),
    low_rank_df.assign(issue="hit_but_low_rank"),
]).drop_duplicates(subset=["query_idx"])

bad_queries_df.drop(columns=["retrieved_passage_ids", "relevant_passage_ids"]).to_csv(
    "benchmark_60q_bad_queries.csv", index=False
)
print(f"\nSaved {len(bad_queries_df)} problem queries -> benchmark_60q_bad_queries.csv")

Total queries: 60
Complete misses (valid GT, not found): 4
Low top1_score (< 0.75): 8
Hit but low rank (found, RR < 0.5): 6

========== COMPLETE MISSES ==========
 query_idx  query_id  query_type                                                             query_en  top1_score                                                  retrieved_passage_ids relevant_passage_ids
         6    362791 DESCRIPTION                              how to get someone to stop smoking weed    0.829517      [hi_362791_5, hi_362791_4, hi_362791_8, hi_362791_7, hi_362791_3]        [hi_362791_0]
        21   1013912      ENTITY which nematodes commonly cause gastrointestinal infections in humans    0.818248 [hi_1013912_3, hi_1013912_4, hi_1013912_9, hi_1013912_0, hi_1013912_2]       [hi_1013912_5]
        49    860160 DESCRIPTION                                                     what is valgrind    0.911293      [hi_860160_4, hi_860160_3, hi_860160_0, hi_860160_2, hi_860160_7]        [hi_860160_6]
        54   

In [61]:
# %% CELL 26-DEFS — Re-define all 4 retrieval functions (run after any kernel restart)

import time
from qdrant_client.models import SparseVector, Prefetch, FusionQuery, Fusion


def retrieve_top_k_dense_only(query_text, top_k=5, model=embed_model, qdrant_client=qdrant_client):
    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    qvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    t1 = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=qvec,
        using="dense",
        limit=top_k,
    )
    t2 = time.perf_counter()

    return {
        "results": response.points,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t2 - t0) * 1000,
    }


def retrieve_top_k_sparse_only(query_text, top_k=5, sparse_model=sparse_model, qdrant_client=qdrant_client):
    t0 = time.perf_counter()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
        using="sparse",
        limit=top_k,
    )
    t2 = time.perf_counter()

    return {
        "results": response.points,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t2 - t0) * 1000,
    }


def retrieve_top_k_hybrid_only(query_text, top_k=5, model=embed_model, sparse_model=sparse_model,
                                qdrant_client=qdrant_client):
    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    dvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        prefetch=[
            Prefetch(query=dvec, using="dense", limit=20),
            Prefetch(
                query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
                using="sparse",
                limit=20,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
    )
    t2 = time.perf_counter()

    return {
        "results": response.points,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t2 - t0) * 1000,
    }


def retrieve_top_k_silent(query_text, top_k=5, model=embed_model, qdrant_client=qdrant_client):
    # same as dense_only — kept separate name to match your original benchmark code
    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    qvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    t1 = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=qvec,
        using="dense",
        limit=top_k,
    )
    t2 = time.perf_counter()

    return {
        "results": response.points,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t2 - t0) * 1000,
    }


# quick sanity check all 4
for name, fn in [("dense", retrieve_top_k_dense_only), ("sparse", retrieve_top_k_sparse_only),
                  ("hybrid", retrieve_top_k_hybrid_only), ("silent", retrieve_top_k_silent)]:
    r = fn("what is Error Free Synonym", top_k=3)
    print(f"{name}: {len(r['results'])} results, top1={r['results'][0].score:.4f}")

dense: 3 results, top1=0.7324
sparse: 3 results, top1=6.5473
hybrid: 3 results, top1=0.8333
silent: 3 results, top1=0.7324


In [73]:
# %% CELL 27 — FULL BENCHMARK: dense vs sparse vs hybrid vs silent (original)

import pandas as pd
import numpy as np

# Map mode name -> retrieval function
RETRIEVAL_FUNCS = {
    "dense":  lambda q: retrieve_top_k_dense_only(q, top_k=TOP_K),
    "sparse": lambda q: retrieve_top_k_sparse_only(q, top_k=TOP_K),
    "hybrid": lambda q: retrieve_top_k_hybrid_only(q, top_k=TOP_K),
    "hybrid_weighted": lambda q: retrieve_top_k_hybrid_weighted(q, top_k=TOP_K),
    "silent": lambda q: retrieve_top_k_silent(q, top_k=TOP_K),  # your original dense-only function
}

def run_full_benchmark(mode, func):
    records = []
    for i, row in test_set.iterrows():
        qid, qtext, qtype = row["query_id"], row["query_en"], row["query_type"]
        relevant_ids = gt_map.get(qid, set())

        r = func(qtext)
        retrieved_ids = [res.payload.get("passage_id") for res in r["results"]]
        scores = [res.score for res in r["results"]]

        hit = any(pid in relevant_ids for pid in retrieved_ids)
        rr = 0.0
        for rank, pid in enumerate(retrieved_ids, 1):
            if pid in relevant_ids:
                rr = 1.0 / rank
                break

        records.append({
            "mode": mode,
            "query_idx": i + 1,
            "query_id": qid,
            "query_type": qtype,
            "query_en": qtext,
            "embedding_ms": r["embedding_ms"],
            "qdrant_ms": r["qdrant_ms"],
            "total_ms": r["total_ms"],
            "top1_score": scores[0] if scores else None,
            "hit@k": hit,
            "reciprocal_rank": rr,
            "has_valid_gt": len(relevant_ids) > 0,
        })
    return pd.DataFrame(records)


# ---------------------------------------------------------
# Run all 4 modes
# ---------------------------------------------------------
all_dfs = []
for mode, func in RETRIEVAL_FUNCS.items():
    print(f"Running benchmark: {mode} ...")
    df_mode = run_full_benchmark(mode, func)
    all_dfs.append(df_mode)
    print(f"  done ({len(df_mode)} queries)")

full_results_df = pd.concat(all_dfs, ignore_index=True)


# ---------------------------------------------------------
# Overall summary per mode
# ---------------------------------------------------------
def summarize(df):
    valid_gt = df[df["has_valid_gt"]]
    return pd.Series({
        "n_queries": len(df),
        "n_valid_gt": len(valid_gt),
        f"hit_rate@{TOP_K}_all": round(df["hit@k"].mean(), 4),
        f"hit_rate@{TOP_K}_valid_gt": round(valid_gt["hit@k"].mean(), 4) if len(valid_gt) else None,
        "MRR_all": round(df["reciprocal_rank"].mean(), 4),
        "MRR_valid_gt": round(valid_gt["reciprocal_rank"].mean(), 4) if len(valid_gt) else None,
        "avg_top1_score": round(df["top1_score"].mean(), 4),
        "p50_total_ms": round(np.percentile(df["total_ms"], 50), 2),
        "p90_total_ms": round(np.percentile(df["total_ms"], 90), 2),
        "p99_total_ms": round(np.percentile(df["total_ms"], 99), 2),
        "mean_total_ms": round(df["total_ms"].mean(), 2),
        "mean_embedding_ms": round(df["embedding_ms"].mean(), 2),
        "mean_qdrant_ms": round(df["qdrant_ms"].mean(), 2),
    })

overall_summary = full_results_df.groupby("mode").apply(summarize).reset_index()
print("\n========== OVERALL: DENSE vs SPARSE vs HYBRID vs SILENT ==========")
print(overall_summary.to_string(index=False))


# ---------------------------------------------------------
# Breakdown by query_type per mode (relevance)
# ---------------------------------------------------------
by_type = full_results_df.groupby(["mode", "query_type"]).agg(
    n=("query_id", "count"),
    hit_rate=("hit@k", "mean"),
    mrr=("reciprocal_rank", "mean"),
    avg_total_ms=("total_ms", "mean"),
).round(4).reset_index()

print("\n========== BY QUERY TYPE (all modes) ==========")
print(by_type.to_string(index=False))


# ---------------------------------------------------------
# Save everything
# ---------------------------------------------------------
full_results_df.to_csv("benchmark_all4_full_details.csv", index=False)
overall_summary.to_csv("benchmark_all4_overall_summary.csv", index=False)
by_type.to_csv("benchmark_all4_by_query_type.csv", index=False)

print("\nSaved:")
print(" - benchmark_all4_full_details.csv")
print(" - benchmark_all4_overall_summary.csv")
print(" - benchmark_all4_by_query_type.csv")

Running benchmark: dense ...
  done (60 queries)
Running benchmark: sparse ...
  done (60 queries)
Running benchmark: hybrid ...
  done (60 queries)
Running benchmark: hybrid_weighted ...
  done (60 queries)
Running benchmark: silent ...
  done (60 queries)

========== OVERALL: DENSE vs SPARSE vs HYBRID vs SILENT ==========
           mode  n_queries  n_valid_gt  hit_rate@5_all  hit_rate@5_valid_gt  MRR_all  MRR_valid_gt  avg_top1_score  p50_total_ms  p90_total_ms  p99_total_ms  mean_total_ms  mean_embedding_ms  mean_qdrant_ms
          dense       60.0        39.0          0.5833               0.8974   0.4200        0.6462          0.8170         65.11        109.23        294.31          81.94              52.02           29.92
         hybrid       60.0        39.0          0.6000               0.9231   0.3747        0.5765          0.7849         75.35         93.41        121.59          74.33              46.89           27.44
hybrid_weighted       60.0        39.0          0.616

D:\Users\Dibyajyoti\Temp\Temp\ipykernel_10200\1786940179.py:83: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overall_summary = full_results_df.groupby("mode").apply(summarize).reset_index()


In [63]:
# %% CELL 28 — Find the 21 queries with NO valid ground truth

test_set["has_valid_gt"] = test_set["query_id"].apply(lambda q: len(gt_map.get(q, set())) > 0)

no_gt_df = test_set[~test_set["has_valid_gt"]].copy()

print(f"Total queries: {len(test_set)}")
print(f"Queries WITH valid ground truth: {test_set['has_valid_gt'].sum()}")
print(f"Queries WITHOUT valid ground truth: {len(no_gt_df)}")

print("\n========== QUERIES WITH NO VALID GROUND TRUTH ==========")
print(no_gt_df[["query_id", "query_type", "query_en"]].to_string(index=False))

no_gt_df[["query_id", "query_type", "query_en"]].to_csv("queries_no_valid_gt.csv", index=False)
print("\nSaved -> queries_no_valid_gt.csv")

Total queries: 60
Queries WITH valid ground truth: 39
Queries WITHOUT valid ground truth: 21

========== QUERIES WITH NO VALID GROUND TRUTH ==========
 query_id  query_type                                                                               query_en
   388153 DESCRIPTION                                             how were the first people to use the flute
   118046 DESCRIPTION                                                                 define assurance grade
   864371 DESCRIPTION                                                              what is your body made of
   922571 DESCRIPTION                               what was the significance of the battle of alamo quizlet
     2923 DESCRIPTION                                                                     Error Free Synonym
   348033 DESCRIPTION                                                          how to cash in a prepaid card
   852191     NUMERIC                                                   what is the ub

In [64]:
# %% CELL 29 — Diagnose why each no-GT query has no ground truth

diagnosis = []
for _, row in no_gt_df.iterrows():
    qid = row["query_id"]
    native_rows = sqlite_df[sqlite_df["query_id"] == qid] if 'sqlite_df' in dir() else pd.DataFrame()
    chunks_for_query = all_chunks[all_chunks["query_id"] == qid]

    diagnosis.append({
        "query_id": qid,
        "query_en": row["query_en"],
        "n_chunks_in_qdrant_data": len(chunks_for_query),
        "n_native_rows": len(native_rows),
        "any_is_selected_true": chunks_for_query["is_selected"].any() if "is_selected" in chunks_for_query.columns else "N/A",
    })

diagnosis_df = pd.DataFrame(diagnosis)
print(diagnosis_df.to_string(index=False))
diagnosis_df.to_csv("no_gt_diagnosis.csv", index=False)

 query_id                                                                               query_en  n_chunks_in_qdrant_data  n_native_rows  any_is_selected_true
   388153                                             how were the first people to use the flute                       10             10                 False
   118046                                                                 define assurance grade                       10             10                 False
   864371                                                              what is your body made of                       10             10                 False
   922571                               what was the significance of the battle of alamo quizlet                       10             10                 False
     2923                                                                     Error Free Synonym                       10             10                 False
   348033                                     

In [65]:
# %% CELL 31 — Run dense, sparse, hybrid on the 21 no-GT queries, save answers + timing

modes_to_run = {
    "dense":  retrieve_top_k_dense_only,
    "sparse": retrieve_top_k_sparse_only,
    "hybrid": retrieve_top_k_hybrid_only,
}

TOP_K_SAVE = 10 # how many results per query to store

records = []
for _, row in no_gt_df.iterrows():
    qid, qtext, qtype = row["query_id"], row["query_en"], row["query_type"]

    for mode_name, func in modes_to_run.items():
        r = func(qtext, top_k=TOP_K_SAVE)
        results = r["results"]

        retrieved_ids = [res.payload.get("passage_id") for res in results]
        retrieved_texts = [res.payload.get("text_en", "") for res in results]
        scores = [round(res.score, 4) for res in results]

        records.append({
            "query_id": qid,
            "query_type": qtype,
            "query_en": qtext,
            "mode": mode_name,
            "top1_passage_id": retrieved_ids[0] if retrieved_ids else None,
            "top1_score": scores[0] if scores else None,
            "top1_answer_text": retrieved_texts[0][:300] if retrieved_texts else None,
            "top5_passage_ids": retrieved_ids,
            "top5_scores": scores,
            "top5_answers": [t[:150] for t in retrieved_texts],
            "embedding_ms": round(r["embedding_ms"], 2),
            "qdrant_ms": round(r["qdrant_ms"], 2),
            "total_ms": round(r["total_ms"], 2),
        })

no_gt_answers_df = pd.DataFrame(records)

print(f"Total rows: {len(no_gt_answers_df)} ({len(no_gt_df)} queries x {len(modes_to_run)} modes)")
print(no_gt_answers_df[["query_id", "query_en", "mode", "top1_score", "total_ms"]].to_string(index=False))

no_gt_answers_df.to_csv("no_gt_queries_dense_sparse_hybrid_answers.csv", index=False)
print("\nSaved -> no_gt_queries_dense_sparse_hybrid_answers.csv")

Total rows: 63 (21 queries x 3 modes)
 query_id                                                                               query_en   mode  top1_score  total_ms
   388153                                             how were the first people to use the flute  dense      0.7768    103.93
   388153                                             how were the first people to use the flute sparse      8.7890     31.82
   388153                                             how were the first people to use the flute hybrid      0.5000     49.39
   118046                                                                 define assurance grade  dense      0.7057     64.71
   118046                                                                 define assurance grade sparse      5.8515     29.46
   118046                                                                 define assurance grade hybrid      0.8333     63.84
   864371                                                              what is y

In [66]:
# %% CELL 33 — Two-layer ID validity check: (1) GT match, (2) native/SQLite integrity match

def validate_result(query_id, retrieved_passage_id):
    """
    Check 1 — Relevance/GT match:
        Is retrieved_passage_id one of the officially correct passages for this query_id?
    Check 2 — Integrity match:
        Does retrieved_passage_id actually exist in the native SQLite lookup,
        AND does its stored query_id match the query_id we searched with?
        (Catches payload corruption / mismatched metadata — a different kind of error
        than "wrong answer retrieved".)
    """
    # --- Check 1: ground truth relevance match ---
    relevant_ids = gt_map.get(query_id, set())
    gt_match = retrieved_passage_id in relevant_ids

    # --- Check 2: native/SQLite integrity match ---
    native_row = sqlite_df[sqlite_df["passage_id"] == retrieved_passage_id]
    if len(native_row) == 0:
        integrity_match = False
        integrity_note = "passage_id not found in native SQLite lookup"
    else:
        stored_query_id = native_row.iloc[0]["query_id"]
        integrity_match = (stored_query_id == query_id)
        integrity_note = "OK" if integrity_match else f"query_id mismatch: expected {query_id}, found {stored_query_id}"

    return gt_match, integrity_match, integrity_note


# Run across all modes/queries you already tested
validation_records = []
for _, row in no_gt_answers_df.iterrows():
    qid = row["query_id"]
    top1_pid = row["top1_passage_id"]

    gt_match, integrity_match, note = validate_result(qid, top1_pid)

    validation_records.append({
        "query_id": qid,
        "query_en": row["query_en"],
        "mode": row["mode"],
        "retrieved_passage_id": top1_pid,
        "gt_match": gt_match,
        "integrity_match": integrity_match,
        "integrity_note": note,
    })

validation_df = pd.DataFrame(validation_records)
print(validation_df.to_string(index=False))

# Flag anything with a genuine integrity problem — this is the one that matters
integrity_issues = validation_df[validation_df["integrity_match"] == False]
print(f"\nIntegrity issues found: {len(integrity_issues)}")
if len(integrity_issues):
    print(integrity_issues.to_string(index=False))

validation_df.to_csv("id_validity_check.csv", index=False)
print("\nSaved -> id_validity_check.csv")

 query_id                                                                               query_en   mode retrieved_passage_id  gt_match  integrity_match                                    integrity_note
   388153                                             how were the first people to use the flute  dense          hi_388153_7     False             True                                                OK
   388153                                             how were the first people to use the flute sparse          hi_362791_7     False            False  query_id mismatch: expected 388153, found 362791
   388153                                             how were the first people to use the flute hybrid          hi_388153_7     False             True                                                OK
   118046                                                                 define assurance grade  dense          hi_118046_7     False             True                                         

In [67]:
# %% CELL 34 — Inspect the two "all modes wrong" queries directly

for qid in [447750, 325563]:
    print(f"\n{'='*80}\nquery_id: {qid}")
    q_row = no_gt_df[no_gt_df["query_id"] == qid]
    print("Query text:", q_row["query_en"].values[0])

    # what SHOULD have been retrievable — this query's own passage pool
    own_chunks = all_chunks[all_chunks["query_id"] == qid][["passage_id", "text_en"]]
    print(f"\nOwn passage pool ({len(own_chunks)} chunks):")
    for _, r in own_chunks.iterrows():
        print(f"  {r['passage_id']}: {r['text_en'][:100]}")


query_id: 447750
Query text: meaning of eria

Own passage pool (10 chunks):
  hi_447750_0: The name Aria is a Teutonic baby name. In Teutonic the meaning of the name Aria is: Intelligence of 
  hi_447750_1: Aria /aria/ [3 sylls.] as a girls' name is of Hebrew origin, and the meaning of Aria is lioness.Ital
  hi_447750_2: Aria is a name with Italian origins. The meaning of the name Aria is solo melody. Aria is also consi
  hi_447750_3: Aria is an uncommon given name for females but a very popular last name for both adults and children
  hi_447750_4: The term, which derives from the Greek and Latin 'aer' (air) first appeared in relation to music in 
  hi_447750_5: Aria as a female given name, Aria (לביאה) means 'lioness' in Aramaic and Hebrew, or 'air' in Italian
  hi_447750_6: The name Aria in Greek mythology is the name of a nymph. It is also an Eastern province of the ancie
  hi_447750_7: Meanings and history of the name Aria: | Edit. Type of light, airy song; very pretty. In opera, 

In [68]:
# %% CELL 35 — Check why query 325563 ignored its own good passages

query_text = "how much negative information can you expect the seller to give you about the business"

# what dense actually returned (outside pool)
r = retrieve_top_k_dense_only(query_text, top_k=5)
print("=== What dense actually retrieved ===")
for res in r["results"]:
    print(f"  {res.payload.get('passage_id')} | score={res.score:.4f} | {res.payload.get('text_en')[:80]}")

# manually score against its OWN pool to see what SHOULD have ranked high
print("\n=== Manual similarity check against own pool (hi_325563_*) ===")
own_texts = all_chunks[all_chunks["query_id"] == 325563]["text_en"].tolist()
own_ids = all_chunks[all_chunks["query_id"] == 325563]["passage_id"].tolist()

query_vec = embed_model.encode(
    f"Represent this sentence for searching relevant passages: {query_text}",
    normalize_embeddings=True
)
own_vecs = embed_model.encode(own_texts, normalize_embeddings=True)

import numpy as np
sims = np.dot(own_vecs, query_vec)
for pid, sim in sorted(zip(own_ids, sims), key=lambda x: -x[1]):
    print(f"  {pid}: sim={sim:.4f}")

=== What dense actually retrieved ===
  hi_600448_1 | score=0.7037 | What constitutes a trade secret depends on the business. It can be a manufacturi
  hi_325563_4 | score=0.6917 | Your buyer or seller can leave Feedback for you as well. Note: Buyers can leave 
  hi_325563_3 | score=0.6878 | And there are some pretty obvious downsides to buying an existing business, and 
  hi_600448_6 | score=0.6714 | You must clearly show that you recognized it in advance as a trade secret.. That
  hi_325563_0 | score=0.6602 | If you've received negative Feedback, we recommend you contact your buyer and wo

=== Manual similarity check against own pool (hi_325563_*) ===
  hi_325563_4: sim=0.6917
  hi_325563_3: sim=0.6878
  hi_325563_0: sim=0.6602
  hi_325563_7: sim=0.6173
  hi_325563_6: sim=0.6006
  hi_325563_8: sim=0.5887
  hi_325563_2: sim=0.5836
  hi_325563_5: sim=0.5586
  hi_325563_9: sim=0.5476
  hi_325563_1: sim=0.5291


In [69]:
# %% CELL 36 — Sample a NEW set of 60 queries (different from the first test_set) and rerun benchmark

# Exclude previously tested query_ids so this is a genuinely new set
previous_query_ids = set(test_set["query_id"].tolist())

# Pool of all available queries with a query_id (from your unified/gt source — adjust if named differently)
# Assuming you have a source dataframe with all unique query_id + query_en + query_type, e.g. hi_norm or unified
candidate_pool = unified.drop_duplicates(subset=["query_id"])[["query_id", "query_en", "query_type"]]
candidate_pool = candidate_pool[~candidate_pool["query_id"].isin(previous_query_ids)]

print(f"Available queries not yet tested: {len(candidate_pool)}")

# Stratified sample of 60, matching your original query_type distribution
new_test_set = stratified_sample(candidate_pool, n_total=60, seed=123)  # different seed = different sample
new_test_set = new_test_set.reset_index(drop=True)

print(f"New test set size: {len(new_test_set)}")
print(new_test_set["query_type"].value_counts())

Available queries not yet tested: 2940
New test set size: 60
query_type
DESCRIPTION    32
NUMERIC        16
ENTITY          5
LOCATION        4
PERSON          3
Name: count, dtype: int64


In [70]:
# %% CELL 37 — Run full dense/sparse/hybrid/silent benchmark on the NEW 60-query set

def run_full_benchmark_v2(mode, func, test_df):
    records = []
    for i, row in test_df.iterrows():
        qid, qtext, qtype = row["query_id"], row["query_en"], row["query_type"]
        relevant_ids = gt_map.get(qid, set())

        r = func(qtext)
        retrieved_ids = [res.payload.get("passage_id") for res in r["results"]]
        scores = [res.score for res in r["results"]]

        hit = any(pid in relevant_ids for pid in retrieved_ids)
        rr = 0.0
        for rank, pid in enumerate(retrieved_ids, 1):
            if pid in relevant_ids:
                rr = 1.0 / rank
                break

        records.append({
            "mode": mode,
            "query_id": qid,
            "query_type": qtype,
            "query_en": qtext,
            "embedding_ms": r["embedding_ms"],
            "qdrant_ms": r["qdrant_ms"],
            "total_ms": r["total_ms"],
            "top1_score": scores[0] if scores else None,
            "hit@k": hit,
            "reciprocal_rank": rr,
            "has_valid_gt": len(relevant_ids) > 0,
        })
    return pd.DataFrame(records)


all_dfs_v2 = []
for mode, func in RETRIEVAL_FUNCS.items():
    print(f"Running new benchmark: {mode} ...")
    df_mode = run_full_benchmark_v2(mode, func, new_test_set)
    all_dfs_v2.append(df_mode)

full_results_v2 = pd.concat(all_dfs_v2, ignore_index=True)

def summarize(df):
    valid_gt = df[df["has_valid_gt"]]
    return pd.Series({
        "n_queries": len(df),
        "n_valid_gt": len(valid_gt),
        "hit_rate@5_valid_gt": round(valid_gt["hit@k"].mean(), 4) if len(valid_gt) else None,
        "MRR_valid_gt": round(valid_gt["reciprocal_rank"].mean(), 4) if len(valid_gt) else None,
        "p50_total_ms": round(np.percentile(df["total_ms"], 50), 2),
        "p90_total_ms": round(np.percentile(df["total_ms"], 90), 2),
        "p99_total_ms": round(np.percentile(df["total_ms"], 99), 2),
        "mean_total_ms": round(df["total_ms"].mean(), 2),
    })

overall_summary_v2 = full_results_v2.groupby("mode").apply(summarize).reset_index()
print("\n========== NEW 60-QUERY SET: DENSE vs SPARSE vs HYBRID vs SILENT ==========")
print(overall_summary_v2.to_string(index=False))

full_results_v2.to_csv("benchmark_v2_60q_details.csv", index=False)
overall_summary_v2.to_csv("benchmark_v2_60q_overall_summary.csv", index=False)
print("\nSaved -> benchmark_v2_60q_details.csv, benchmark_v2_60q_overall_summary.csv")

Running new benchmark: dense ...
Running new benchmark: sparse ...
Running new benchmark: hybrid ...
Running new benchmark: silent ...

========== NEW 60-QUERY SET: DENSE vs SPARSE vs HYBRID vs SILENT ==========
  mode  n_queries  n_valid_gt  hit_rate@5_valid_gt  MRR_valid_gt  p50_total_ms  p90_total_ms  p99_total_ms  mean_total_ms
 dense       60.0        41.0               0.8780        0.6264         76.67        120.66        449.20          93.84
hybrid       60.0        41.0               0.8293        0.5220         68.64         78.84         89.55          68.16
silent       60.0        41.0               0.8780        0.6264         67.88         81.89         95.49          69.62
sparse       60.0        41.0               0.6341        0.3711         30.54         32.28         34.14          28.44

Saved -> benchmark_v2_60q_details.csv, benchmark_v2_60q_overall_summary.csv


D:\Users\Dibyajyoti\Temp\Temp\ipykernel_10200\981532876.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overall_summary_v2 = full_results_v2.groupby("mode").apply(summarize).reset_index()


In [71]:
# %% CELL 38 — Hybrid with REDUCED sparse influence (smaller prefetch limit)

def retrieve_top_k_hybrid_weighted_v1(query_text, top_k=5, model=embed_model,
                                        sparse_model=sparse_model, qdrant_client=qdrant_client):
    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    dvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    response = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        prefetch=[
            Prefetch(query=dvec, using="dense", limit=30),   # dense gets MORE candidates
            Prefetch(
                query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
                using="sparse",
                limit=10,   # sparse gets FEWER candidates -> less influence in RRF
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
    )
    t2 = time.perf_counter()

    return {
        "results": response.points,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t2 - t0) * 1000,
    }

In [72]:
# %% CELL 39 — Manual weighted fusion (dense-dominant, tunable)

def retrieve_top_k_hybrid_weighted(query_text, top_k=5, dense_weight=0.75, sparse_weight=0.25,
                                     candidate_pool=20, model=embed_model,
                                     sparse_model=sparse_model, qdrant_client=qdrant_client):
    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    dvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    # get separate candidate lists from each retriever
    dense_resp = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION, query=dvec, using="dense", limit=candidate_pool
    )
    sparse_resp = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
        using="sparse", limit=candidate_pool
    )
    t2 = time.perf_counter()

    # normalize each score list to 0-1 range so weighting is meaningful
    def normalize(points):
        scores = [p.score for p in points]
        if not scores:
            return {}
        lo, hi = min(scores), max(scores)
        rng = (hi - lo) or 1.0
        return {p.payload.get("passage_id"): (p.score - lo) / rng for p in points}

    dense_norm = normalize(dense_resp.points)
    sparse_norm = normalize(sparse_resp.points)

    # build lookup of full point objects (prefer dense's payload, fallback to sparse's)
    point_lookup = {p.payload.get("passage_id"): p for p in dense_resp.points}
    for p in sparse_resp.points:
        point_lookup.setdefault(p.payload.get("passage_id"), p)

    # weighted combine
    all_ids = set(dense_norm) | set(sparse_norm)
    fused_scores = {
        pid: dense_weight * dense_norm.get(pid, 0.0) + sparse_weight * sparse_norm.get(pid, 0.0)
        for pid in all_ids
    }

    ranked_ids = sorted(fused_scores, key=lambda pid: fused_scores[pid], reverse=True)[:top_k]

    # build result objects with fused score attached
    results = []
    for pid in ranked_ids:
        p = point_lookup[pid]
        p.score = fused_scores[pid]  # overwrite with fused score
        results.append(p)

    t3 = time.perf_counter()

    return {
        "results": results,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "fusion_ms": (t3 - t2) * 1000,
        "dedup_ms": 0.0,
        "total_ms": (t3 - t0) * 1000,
    }


# sanity check
r = retrieve_top_k_hybrid_weighted("what is Error Free Synonym", top_k=5)
for i, res in enumerate(r["results"], 1):
    print(f"{i}. fused_score={res.score:.4f} | {res.payload.get('text_en')[:100]}")

1. fused_score=0.9504 | Synonym Discussion of ERROR. error, mistake, and blunder mean an act or statement that is not right 
2. fused_score=0.5325 | 1 First, present your prospective employer with a concise, error-free resume. 2 The editors are to b
3. fused_score=0.2148 | 1 part of a statement that is not correct 1. 2 the book was full of errors 1. ( 3 computer science) 
4. fused_score=0.2078 | Shush.se - Watch The Office Season 8 Episode: 5 – Spooked | Free Online Streaming |. Shush.se - Watc
5. fused_score=0.2006 | uk ​ /ˈmʌm.bəl/ us ​ /ˈmʌm.bəl/. B2 to speak quietly and in a way that is not clear so that the word


In [74]:
# %% CELL 40 — ID integrity check across ALL modes, full 60-query set

def validate_result(query_id, retrieved_passage_id):
    relevant_ids = gt_map.get(query_id, set())
    gt_match = retrieved_passage_id in relevant_ids

    native_row = sqlite_df[sqlite_df["passage_id"] == retrieved_passage_id]
    if len(native_row) == 0:
        integrity_match = False
        note = "passage_id not found in native SQLite lookup"
    else:
        stored_query_id = native_row.iloc[0]["query_id"]
        integrity_match = (stored_query_id == query_id)
        note = "OK" if integrity_match else f"query_id mismatch: expected {query_id}, found {stored_query_id}"

    return gt_match, integrity_match, note


integrity_records = []
for _, row in full_results_df.iterrows():   # your full 60-query x 5-mode results
    qid = row["query_id"]
    mode = row["mode"]

    # re-fetch top1 passage_id for this query+mode combo
    r = RETRIEVAL_FUNCS[mode](row["query_en"]) if mode in RETRIEVAL_FUNCS else None
    if r is None:
        continue
    retrieved = r["results"]
    if not retrieved:
        continue
    top1_pid = retrieved[0].payload.get("passage_id")

    gt_match, integrity_match, note = validate_result(qid, top1_pid)

    integrity_records.append({
        "query_id": qid,
        "mode": mode,
        "query_en": row["query_en"],
        "retrieved_passage_id": top1_pid,
        "gt_match": gt_match,
        "integrity_match": integrity_match,
        "note": note,
    })

integrity_df = pd.DataFrame(integrity_records)

issues = integrity_df[integrity_df["integrity_match"] == False]
print(f"Total checked: {len(integrity_df)}")
print(f"Integrity issues found: {len(issues)}")
if len(issues):
    print(issues.to_string(index=False))

# breakdown by mode — this tells you if hybrid_weighted specifically introduced problems
issues_by_mode = integrity_df.groupby("mode")["integrity_match"].apply(lambda s: (~s).sum()).reset_index(name="n_issues")
print("\nIntegrity issues by mode:")
print(issues_by_mode.to_string(index=False))

integrity_df.to_csv("integrity_check_all_modes_60q.csv", index=False)
print("\nSaved -> integrity_check_all_modes_60q.csv")

Total checked: 300
Integrity issues found: 22
 query_id            mode                                                                                  query_en retrieved_passage_id  gt_match  integrity_match                                              note
   447750           dense                                                                           meaning of eria          hi_911316_0     False            False  query_id mismatch: expected 447750, found 911316
  1146530           dense                                                                what regulation covers sop          hi_717293_5     False            False query_id mismatch: expected 1146530, found 717293
   325563           dense    how much negative information can you expect the seller to give you about the business          hi_600448_1     False            False  query_id mismatch: expected 325563, found 600448
   388153          sparse                                                how were the first people

In [75]:
# %% CELL 41 — Inspect 1146530's own passage pool

qid = 1146530
own_chunks = all_chunks[all_chunks["query_id"] == qid][["passage_id", "text_en"]]
print(f"Query: what regulation covers sop")
print(f"\nOwn passage pool ({len(own_chunks)} chunks):")
for _, r in own_chunks.iterrows():
    print(f"  {r['passage_id']}: {r['text_en'][:120]}")

Query: what regulation covers sop

Own passage pool (10 chunks):
  hi_1146530_0: III CORPS & FORT HOOD REGULATION 750-2 ● 5 APRIL 2012 9. (6) BSB. The BSB is an organic unit of the Brigade Combat Team 
  hi_1146530_1: 2 III CORPS & FORT HOOD REGULATION 750-2  5 APRIL 2012. 13th ESC will set policy for pass back operations which may inc
  hi_1146530_2: The purpose of this regulation is to provide comprehensive guidance to the Army National Guard on the recruiting and ret
  hi_1146530_3: III Corps and Fort Hood Regulation 750-2 Maintenance Policies and Procedures This issue dated 5 April 2012 This is a maj
  hi_1146530_4: This regulation is a consolidation of NGR 601-1 and NGR 601-2 that covers the Army National Guard Strength Maintenance P
  hi_1146530_5: Abbreviations and special terms used in this regulation are explained in the glossary. In addition, the use of the term 
  hi_1146530_6: The purpose of this regulation is to provide comprehensive guidance to the Army National Guard on

In [76]:
# %% CELL 42 — Check why 1146530 ignored its own pool

query_text = "what regulation covers sop"

r = retrieve_top_k_dense_only(query_text, top_k=5)
print("=== What dense actually retrieved ===")
for res in r["results"]:
    print(f"  {res.payload.get('passage_id')} | score={res.score:.4f} | {res.payload.get('text_en')[:100]}")

print("\n=== Manual similarity against own pool (hi_1146530_*) ===")
own_texts = all_chunks[all_chunks["query_id"] == 1146530]["text_en"].tolist()
own_ids = all_chunks[all_chunks["query_id"] == 1146530]["passage_id"].tolist()

query_vec = embed_model.encode(
    f"Represent this sentence for searching relevant passages: {query_text}",
    normalize_embeddings=True
)
own_vecs = embed_model.encode(own_texts, normalize_embeddings=True)

sims = np.dot(own_vecs, query_vec)
for pid, sim in sorted(zip(own_ids, sims), key=lambda x: -x[1]):
    print(f"  {pid}: sim={sim:.4f}")

=== What dense actually retrieved ===
  hi_717293_5 | score=0.6839 | Tuesday, April 7th 2016. Renaud Anjoran, the founder of Sofeast in Shenzhen, is a well-known expert 
  hi_1069213_3 | score=0.6788 | These inputs form the basis of policy frameworks that influence solid waste management decisions. In
  hi_1069213_1 | score=0.6701 | Proper Waste Disposal and the U.S. Government. The majority of the laws associated with waste dispos
  hi_616244_1 | score=0.6688 | TRID Rule to Take Effect October 3, 2015. The CFPB announced today it is delaying the effective date
  hi_1015762_7 | score=0.6666 | The right of rescission is a consumer protection law found within the Truth in Lending Act. The Trut

=== Manual similarity against own pool (hi_1146530_*) ===
  hi_1146530_6: sim=0.6613
  hi_1146530_8: sim=0.6598
  hi_1146530_5: sim=0.6595
  hi_1146530_4: sim=0.6532
  hi_1146530_1: sim=0.6327
  hi_1146530_7: sim=0.6099
  hi_1146530_3: sim=0.5965
  hi_1146530_0: sim=0.5894
  hi_1146530_2: sim=0.57

In [78]:
# %% CELL 43 — Test acronym expansion: original vs full-form query, across all 5 modes

test_queries = {
    1146530: {
        "original": "what regulation covers sop",
        "expanded": "what regulation covers SOP (Standard Operating Procedure)",
    },
    447750: {
        "original": "meaning of eria",
        "expanded": "meaning of aria",   # correcting the likely dataset typo
    },
    325563: {
        "original": "how much negative information can you expect the seller to give you about the business",
        "expanded": "how much negative feedback information can you expect the seller to disclose about the business",
    },
}

expansion_records = []

for qid, variants in test_queries.items():
    relevant_ids = gt_map.get(qid, set())
    own_passage_ids = set(all_chunks[all_chunks["query_id"] == qid]["passage_id"].tolist())

    for variant_name, qtext in variants.items():
        for mode, func in RETRIEVAL_FUNCS.items():
            r = func(qtext)
            results = r["results"]
            retrieved_ids = [res.payload.get("passage_id") for res in results]
            scores = [res.score for res in results]

            top1_pid = retrieved_ids[0] if retrieved_ids else None
            top1_score = scores[0] if scores else None

            in_own_pool = top1_pid in own_passage_ids if top1_pid else False
            gt_hit = any(pid in relevant_ids for pid in retrieved_ids)

            expansion_records.append({
                "query_id": qid,
                "variant": variant_name,
                "query_text": qtext,
                "mode": mode,
                "top1_passage_id": top1_pid,
                "top1_score": round(top1_score, 4) if top1_score else None,
                "top1_answer": results[0].payload.get("text_en", "")[:100] if results else None,
                "retrieved_from_own_pool": in_own_pool,
                "gt_hit": gt_hit,
                "total_ms": round(r["total_ms"], 2),
            })

expansion_df = pd.DataFrame(expansion_records)
print(expansion_df.to_string(index=False))

expansion_df.to_csv("acronym_expansion_test.csv", index=False)
print("\nSaved -> acronym_expansion_test.csv")

 query_id  variant                                                                                      query_text            mode top1_passage_id  top1_score                                                                                          top1_answer  retrieved_from_own_pool  gt_hit  total_ms
  1146530 original                                                                      what regulation covers sop           dense     hi_717293_5      0.6839 Tuesday, April 7th 2016. Renaud Anjoran, the founder of Sofeast in Shenzhen, is a well-known expert                     False   False     68.35
  1146530 original                                                                      what regulation covers sop          sparse    hi_1146530_4      5.6653 This regulation is a consolidation of NGR 601-1 and NGR 601-2 that covers the Army National Guard St                     True   False     28.16
  1146530 original                                                                      wha

In [81]:
# %% CELL 45 — Manual RRF with tunable k

def retrieve_top_k_rrf_tunable(query_text, top_k=5, k=60, candidate_pool=20,
                                 model=embed_model, sparse_model=sparse_model,
                                 qdrant_client=qdrant_client):
    t0 = time.perf_counter()
    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    dvec = model.encode(prefixed, normalize_embeddings=True).tolist()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    dense_resp = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION, query=dvec, using="dense", limit=candidate_pool
    )
    sparse_resp = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
        using="sparse", limit=candidate_pool
    )
    t2 = time.perf_counter()

    rrf_scores = {}
    point_lookup = {}
    for rank, p in enumerate(dense_resp.points, 1):
        pid = p.payload.get("passage_id")
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1 / (k + rank)
        point_lookup[pid] = p
    for rank, p in enumerate(sparse_resp.points, 1):
        pid = p.payload.get("passage_id")
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1 / (k + rank)
        point_lookup.setdefault(pid, p)

    ranked = sorted(rrf_scores, key=lambda pid: rrf_scores[pid], reverse=True)[:top_k]
    results = []
    for pid in ranked:
        p = point_lookup[pid]
        p.score = rrf_scores[pid]
        results.append(p)

    t3 = time.perf_counter()
    return {
        "results": results,
        "embedding_ms": (t1 - t0) * 1000,
        "qdrant_ms": (t2 - t1) * 1000,
        "fusion_ms": (t3 - t2) * 1000,
        "total_ms": (t3 - t0) * 1000,
    }

In [82]:
# %% CELL 46 — FULL CLEAN BENCHMARK: every mode, one table, relevance + timing

import time
import numpy as np
import pandas as pd
from qdrant_client.models import SparseVector, Prefetch, FusionQuery, Fusion

TOP_K = 5

# ---------------------------------------------------------
# 1. Register every mode to test
# ---------------------------------------------------------
ALL_MODES = {
    "dense":           lambda q: retrieve_top_k_dense_only(q, top_k=TOP_K),
    "sparse":          lambda q: retrieve_top_k_sparse_only(q, top_k=TOP_K),
    "hybrid_rrf_k60":  lambda q: retrieve_top_k_hybrid_only(q, top_k=TOP_K),          # Qdrant native RRF (fixed k)
    "hybrid_weighted": lambda q: retrieve_top_k_hybrid_weighted(q, top_k=TOP_K),      # your 0.75/0.25 fix
    "rrf_k5":          lambda q: retrieve_top_k_rrf_tunable(q, top_k=TOP_K, k=5),
    "rrf_k10":         lambda q: retrieve_top_k_rrf_tunable(q, top_k=TOP_K, k=10),
    "rrf_k30":         lambda q: retrieve_top_k_rrf_tunable(q, top_k=TOP_K, k=30),
    "rrf_k100":        lambda q: retrieve_top_k_rrf_tunable(q, top_k=TOP_K, k=100),
}

# ---------------------------------------------------------
# 2. Run benchmark for one mode
# ---------------------------------------------------------
def run_mode(mode_name, func, test_df):
    records = []
    for _, row in test_df.iterrows():
        qid, qtext, qtype = row["query_id"], row["query_en"], row["query_type"]
        relevant_ids = gt_map.get(qid, set())

        r = func(qtext)
        retrieved_ids = [res.payload.get("passage_id") for res in r["results"]]
        scores = [res.score for res in r["results"]]

        hit = any(pid in relevant_ids for pid in retrieved_ids)
        rr = 0.0
        for rank, pid in enumerate(retrieved_ids, 1):
            if pid in relevant_ids:
                rr = 1.0 / rank
                break

        records.append({
            "mode": mode_name,
            "query_id": qid,
            "query_type": qtype,
            "hit@5": hit,
            "rr": rr,
            "top1_score": scores[0] if scores else None,
            "total_ms": r["total_ms"],
            "has_valid_gt": len(relevant_ids) > 0,
        })
    return pd.DataFrame(records)


# ---------------------------------------------------------
# 3. Run everything
# ---------------------------------------------------------
all_dfs = []
for name, func in ALL_MODES.items():
    print(f"Running: {name} ...")
    all_dfs.append(run_mode(name, func, test_set))

full_bench_df = pd.concat(all_dfs, ignore_index=True)


# ---------------------------------------------------------
# 4. Clean summary table
# ---------------------------------------------------------
def summarize(df):
    valid = df[df["has_valid_gt"]]
    return pd.Series({
        "hit_rate@5": round(valid["hit@5"].mean(), 4),
        "MRR": round(valid["rr"].mean(), 4),
        "p50_ms": round(np.percentile(df["total_ms"], 50), 1),
        "p90_ms": round(np.percentile(df["total_ms"], 90), 1),
        "p99_ms": round(np.percentile(df["total_ms"], 99), 1),
        "mean_ms": round(df["total_ms"].mean(), 1),
    })

clean_summary = full_bench_df.groupby("mode").apply(summarize).reset_index()
clean_summary = clean_summary.sort_values("hit_rate@5", ascending=False)

print("\n" + "="*90)
print("FULL BENCHMARK — ALL MODES (39 queries with valid ground truth)")
print("="*90)
print(clean_summary.to_string(index=False))

clean_summary.to_csv("FULL_benchmark_all_modes.csv", index=False)
full_bench_df.to_csv("FULL_benchmark_all_modes_details.csv", index=False)
print("\nSaved -> FULL_benchmark_all_modes.csv, FULL_benchmark_all_modes_details.csv")

Running: dense ...
Running: sparse ...
Running: hybrid_rrf_k60 ...
Running: hybrid_weighted ...
Running: rrf_k5 ...
Running: rrf_k10 ...
Running: rrf_k30 ...
Running: rrf_k100 ...

FULL BENCHMARK — ALL MODES (39 queries with valid ground truth)
           mode  hit_rate@5    MRR  p50_ms  p90_ms  p99_ms  mean_ms
hybrid_weighted      0.9487 0.6115    93.2   108.9   129.1     93.6
 hybrid_rrf_k60      0.9231 0.5274    64.5    82.3   101.4     69.1
          dense      0.8974 0.6462    63.3   121.2   335.4     84.9
         rrf_k5      0.8718 0.5274    95.4   113.0   130.9     97.8
        rrf_k10      0.8462 0.5026   105.4   127.1   140.2    103.1
       rrf_k100      0.8205 0.4919    91.6   110.7   118.1     90.4
        rrf_k30      0.8205 0.4919    93.3   110.1   127.1     91.3
         sparse      0.7179 0.4085    30.0    32.6   458.5     44.3

Saved -> FULL_benchmark_all_modes.csv, FULL_benchmark_all_modes_details.csv


D:\Users\Dibyajyoti\Temp\Temp\ipykernel_10200\3654961307.py:82: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  clean_summary = full_bench_df.groupby("mode").apply(summarize).reset_index()


In [83]:
sparse_slow = full_bench_df[full_bench_df["mode"] == "sparse"].nlargest(3, "total_ms")
print(sparse_slow[["query_id", "total_ms"]])

    query_id   total_ms
61    388153  1068.1873
99    427189    34.7710
76    348033    34.1296


In [84]:
# %% CELL 47 — Re-run just this one query a few times to check if it's reproducible

query_text = test_set[test_set["query_id"] == 388153]["query_en"].values[0]
print(f"Query: {query_text}\n")

for i in range(5):
    r = retrieve_top_k_sparse_only(query_text, top_k=5)
    print(f"Run {i+1}: total_ms = {r['total_ms']:.2f}")

Query: how were the first people to use the flute

Run 1: total_ms = 35.25
Run 2: total_ms = 29.28
Run 3: total_ms = 29.12
Run 4: total_ms = 36.08
Run 5: total_ms = 28.55


In [86]:
# %% CELL 48 — FULL BENCHMARK + ID INTEGRITY CHECK, all modes, one report

import time
import numpy as np
import pandas as pd

def validate_result(query_id, retrieved_passage_id):
    """Check 1: GT relevance match. Check 2: native SQLite ID integrity match."""
    relevant_ids = gt_map.get(query_id, set())
    gt_match = retrieved_passage_id in relevant_ids

    native_row = sqlite_df[sqlite_df["passage_id"] == retrieved_passage_id]
    if len(native_row) == 0:
        integrity_match = False
        note = "passage_id not found in native SQLite lookup"
    else:
        stored_query_id = native_row.iloc[0]["query_id"]
        integrity_match = (stored_query_id == query_id)
        note = "OK" if integrity_match else f"query_id mismatch: expected {query_id}, found {stored_query_id}"

    return gt_match, integrity_match, note


def run_mode_with_integrity(mode_name, func, test_df):
    records = []
    for _, row in test_df.iterrows():
        qid, qtext, qtype = row["query_id"], row["query_en"], row["query_type"]
        relevant_ids = gt_map.get(qid, set())

        r = func(qtext)
        retrieved_ids = [res.payload.get("passage_id") for res in r["results"]]
        scores = [res.score for res in r["results"]]

        hit = any(pid in relevant_ids for pid in retrieved_ids)
        rr = 0.0
        for rank, pid in enumerate(retrieved_ids, 1):
            if pid in relevant_ids:
                rr = 1.0 / rank
                break

        top1_pid = retrieved_ids[0] if retrieved_ids else None
        gt_match, integrity_match, note = (False, None, "no results") if top1_pid is None \
            else validate_result(qid, top1_pid)

        records.append({
            "mode": mode_name,
            "query_id": qid,
            "query_type": qtype,
            "query_en": qtext,
            "top1_passage_id": top1_pid,
            "top1_score": scores[0] if scores else None,
            "hit@5": hit,
            "rr": rr,
            "gt_match_top1": gt_match,
            "integrity_match": integrity_match,
            "integrity_note": note,
            "total_ms": r["total_ms"],
            "has_valid_gt": len(relevant_ids) > 0,
        })
    return pd.DataFrame(records)


# ---------------------------------------------------------
# Run all modes
# ---------------------------------------------------------
all_dfs = []
for name, func in ALL_MODES.items():
    print(f"Running: {name} ...")
    all_dfs.append(run_mode_with_integrity(name, func, test_set))

full_report_df = pd.concat(all_dfs, ignore_index=True)


# ---------------------------------------------------------
# Relevance + timing summary
# ---------------------------------------------------------
def summarize(df):
    valid = df[df["has_valid_gt"]]
    return pd.Series({
        "hit_rate@5": round(valid["hit@5"].mean(), 4),
        "MRR": round(valid["rr"].mean(), 4),
        "mean_ms": round(df["total_ms"].mean(), 1),
        "p90_ms": round(np.percentile(df["total_ms"], 90), 1),
    })

perf_summary = full_report_df.groupby("mode").apply(summarize).reset_index()

# ---------------------------------------------------------
# ID integrity summary — the new part
# ---------------------------------------------------------
integrity_summary = full_report_df.groupby("mode").agg(
    n_queries=("query_id", "count"),
    n_integrity_issues=("integrity_match", lambda s: (s == False).sum()),
).reset_index()
integrity_summary["integrity_issue_rate"] = round(
    integrity_summary["n_integrity_issues"] / integrity_summary["n_queries"], 4
)

# ---------------------------------------------------------
# Merge into one final report table
# ---------------------------------------------------------
final_report = perf_summary.merge(integrity_summary, on="mode").sort_values("hit_rate@5", ascending=False)

print("\n" + "="*100)
print("FULL BENCHMARK + ID INTEGRITY REPORT — ALL MODES")
print("="*100)
print(final_report.to_string(index=False))

# ---------------------------------------------------------
# List the actual integrity issues found, for inspection
# ---------------------------------------------------------
issues_df = full_report_df[full_report_df["integrity_match"] == False]
print(f"\nTotal integrity issues across all modes: {len(issues_df)}")
if len(issues_df):
    print(issues_df[["mode", "query_id", "query_en", "top1_passage_id", "integrity_note"]].to_string(index=False))

# ---------------------------------------------------------
# Save everything
# ---------------------------------------------------------
final_report.to_csv("FINAL_report_summary.csv", index=False)
full_report_df.to_csv("FINAL_report_full_details.csv", index=False)
issues_df.to_csv("FINAL_report_integrity_issues.csv", index=False)

print("\nSaved:")
print(" - FINAL_report_summary.csv        (one row per mode: hit_rate, MRR, timing, integrity)")
print(" - FINAL_report_full_details.csv   (every query x mode)")
print(" - FINAL_report_integrity_issues.csv (only the flagged mismatches)")

Running: dense ...
Running: sparse ...
Running: hybrid_rrf_k60 ...
Running: hybrid_weighted ...
Running: rrf_k5 ...
Running: rrf_k10 ...
Running: rrf_k30 ...
Running: rrf_k100 ...

FULL BENCHMARK + ID INTEGRITY REPORT — ALL MODES
           mode  hit_rate@5    MRR  mean_ms  p90_ms  n_queries  n_integrity_issues  integrity_issue_rate
hybrid_weighted      0.9487 0.6115     91.8   107.8         60                   3                0.0500
 hybrid_rrf_k60      0.9231 0.5658     58.9    72.6         60                   5                0.0833
          dense      0.8974 0.6462     70.0    98.1         60                   3                0.0500
         rrf_k5      0.8718 0.5274     86.9   103.4         60                   4                0.0667
        rrf_k10      0.8462 0.5026    105.2   133.3         60                   4                0.0667
       rrf_k100      0.8205 0.4919     88.1   104.2         60                   5                0.0833
        rrf_k30      0.8205 0.4919 

D:\Users\Dibyajyoti\Temp\Temp\ipykernel_10200\1696427144.py:86: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  perf_summary = full_report_df.groupby("mode").apply(summarize).reset_index()


In [87]:
# %% CELL 49 — Check the 2 remaining unverified multi-mode failures

for qid, qtext in [(503650, "steps to calcukate RBRVS"),
                    (427189, "is the process through which a tumor supports its growth by creating its own blood supply")]:
    print(f"\n{'='*80}\nquery_id: {qid} | {qtext}")

    r = retrieve_top_k_dense_only(qtext, top_k=3)
    print("Top dense result:", r["results"][0].payload.get("text_en")[:150])

    own_texts = all_chunks[all_chunks["query_id"] == qid]["text_en"].tolist()
    own_ids = all_chunks[all_chunks["query_id"] == qid]["passage_id"].tolist()
    qvec = embed_model.encode(f"Represent this sentence for searching relevant passages: {qtext}", normalize_embeddings=True)
    own_vecs = embed_model.encode(own_texts, normalize_embeddings=True)
    sims = np.dot(own_vecs, qvec)
    for pid, sim in sorted(zip(own_ids, sims), key=lambda x: -x[1])[:3]:
        print(f"  {pid}: sim={sim:.4f}")


query_id: 503650 | steps to calcukate RBRVS
Top dense result: Free RBRVS Calculator For Your Medical Practice. My friends over at Pedsource just published a free RBRVS Calculator based on 2012 data. According to 
  hi_503650_7: sim=0.6873
  hi_503650_6: sim=0.6331
  hi_503650_2: sim=0.6133

query_id: 427189 | is the process through which a tumor supports its growth by creating its own blood supply
Top dense result: So they send out signals, called angiogenic factors, that encourage new blood vessels to grow into the tumour. This is called angiogenesis. Without a 
  hi_427189_3: sim=0.8009
  hi_427189_2: sim=0.7525
  hi_427189_7: sim=0.7046


In [88]:
# %% CELL 50 — What did dense actually return for 427189, and where is it from?

r = retrieve_top_k_dense_only(
    "is the process through which a tumor supports its growth by creating its own blood supply", top_k=3
)
for res in r["results"]:
    print(f"{res.payload.get('passage_id')} | score={res.score:.4f} | query_id={res.payload.get('query_id')}")
    print(f"  {res.payload.get('text_en')[:150]}\n")

hi_427189_3 | score=0.8009 | query_id=427189
  So they send out signals, called angiogenic factors, that encourage new blood vessels to grow into the tumour. This is called angiogenesis. Without a 

hi_427189_2 | score=0.7525 | query_id=427189
  So now you know how cancer spreads around the body and also how it is actually difficult to do so. Metastasis' refers to the spread of cancer cells ar

hi_1010589_5 | score=0.7258 | query_id=1010589
  1 Damaged cells replicate, creating more damaged cells and tumor growth. 2 Our body’s hormones and chemicals can accelerate the growth of some tumors.

